In [ ]:
!pip uninstall -y docling docling-core docling-slim docling-parse docling-ibm-models

In [ ]:
!pip install -q "git+https://github.com/docling-project/docling.git"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 279.0/279.0 kB 2.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.2/79.2 kB 4.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.


In [ ]:
!pip install -q pydantic rich pdf2image pillow docling-parse \
               docling-ibm-models pypdfium2 pylatexenc \
               marko python-pptx python-docx qwen-vl-utils

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 873.8 kB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 2.3 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 32.5 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.0/94.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 94.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.7/42.7 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.8/472.8 kB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 63.4 MB/s eta 0:00:0000:010:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 661.5/661.5 kB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import re
import json
from pydantic import BaseModel, Field
from rich import print
from typing import Optional, List
from docling.document_extractor import DocumentExtractor
from docling.datamodel.base_models import InputFormat
from docling.document_converter import DocumentConverter, PdfFormatOption, ImageFormatOption
from docling.pipeline.vlm_pipeline import VlmPipeline

# **Task 1: Information Extraction with a VLM**

#### **reciept.png**

* Pydantic Models:

In [ ]:
# Invoice Items
class InvoiceItem(BaseModel):
    item_name: str = Field(description="The name of the product")
    quantity: float = Field(description="The quantity purchased")
    unit_price: float = Field(description="Price per item")
    amount: float = Field(description="Total amount for this item")

# Billing Information
class BillingInfo(BaseModel):
    customer_name: str = Field(description="Customer or vendor name")
    invoice_number: str = Field(description="Invoice number")
    invoice_date: str = Field(description="Invoice date")

# Payment Summary
class PaymentSummary(BaseModel):
    subtotal: float = Field(description="Subtotal before tax")
    tax: float = Field(description="Tax amount")
    total: float = Field(description="Final invoice total")

# Main Invoice
class InvoiceData(BaseModel):
    billing_info: BillingInfo
    items: List[InvoiceItem]
    payment_summary: PaymentSummary

* Extract and convert document using the Docling Vision Language Model (VLM) pipeline:

In [ ]:
source = "/content/invoice.jpg"

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_cls=VlmPipeline
        ),
        InputFormat.IMAGE: ImageFormatOption(
            pipeline_cls=VlmPipeline
        ),
    }
)

result = converter.convert(source).document
markdown_output = result.export_to_markdown()
print("MARKDOWN OUTPUT:\n", markdown_output)

Loading weights:   0%|          | 0/470 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'use_cache', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


MARKDOWN OUTPUT:
 <!-- image -->

## BILLED TO:

Imani Olowe +123-456-7890 63 Ivy Road, Hawkville, GA, USA 31036

Invoice No. 12345 16 June 2025

| Item                  | Quantity   | Unit Price   | Total   |
|-----------------------|------------|--------------|---------|
| Eggshell Camisole Top | 1          | $123         | $123    |
| Cuban Collar Shirt    | 2          | $127         | $254    |
| Floral Cotton Dress   | 1          | $123         | $123    |
|                       |            | Subtotal     | $500    |
|                       |            | Tax (0%)     | $0      |
|                       |            | Total        | $500    |

## Thank you!

## PAYMENT INFORMATION

Briard Bank Account Name: Samira Hadid Account No.: 123-456-7890 Pay by: 5 July 2025

<!-- image -->

* Print The Structured Output:

In [ ]:
text = markdown_output

# Invoice number + date

invoice_match = re.search(r"Invoice No\.\s*(\d+)\s*(.+)", text)
invoice_number = invoice_match.group(1)
invoice_date = invoice_match.group(2).strip()

# Vendor / Customer Name

customer_name = (text.split("## BILLED TO:")[1].split("\n")[2].split("+")[0].strip())

# Extract Totals

subtotal = float(re.search(r"Subtotal\s*\|\s*\$(\d+)", text).group(1))
tax = float(re.search(r"Tax.*\|\s*\$(\d+)", text).group(1))
total = float(re.search(r"Total\s*\|\s*\$(\d+)", text).group(1))

# Extract Invoice Items

items = []
rows = re.findall(r"\|\s*([A-Za-z ]+)\s*\|\s*(\d+)\s*\|\s*\$(\d+)\s*\|\s*\$(\d+)", text)

for row in rows:
    items.append(
        InvoiceItem(
            item_name=row[0].strip(),
            quantity=float(row[1]),
            unit_price=float(row[2]),
            amount=float(row[3])
        )
    )

# Create Final JSON Object

invoice_data = InvoiceData(
    billing_info=BillingInfo(
        customer_name=customer_name,
        invoice_number=invoice_number,
        invoice_date=invoice_date,
    ),

    items=items,

    payment_summary=PaymentSummary(
        subtotal=subtotal,
        tax=tax,
        total=total,
    )
)

# Print Final JSON

print("\n========= STRUCTURED OUTPUT ==========\n")
print(invoice_data.model_dump_json(indent=2))

========= STRUCTURED OUTPUT ==========

{
  "billing_info": {
    "customer_name": "Imani Olowe",
    "invoice_number": "12345",
    "invoice_date": "16 June 2025"
  },
  "items": [
    {
      "item_name": "Eggshell Camisole Top",
      "quantity": 1.0,
      "unit_price": 123.0,
      "amount": 123.0
    },
    {
      "item_name": "Cuban Collar Shirt",
      "quantity": 2.0,
      "unit_price": 127.0,
      "amount": 254.0
    },
    {
      "item_name": "Floral Cotton Dress",
      "quantity": 1.0,
      "unit_price": 123.0,
      "amount": 123.0
    }
  ],
  "payment_summary": {
    "subtotal": 500.0,
    "tax": 0.0,
    "total": 500.0
  }
}

#### **resume.pdf**

* Pydantic Models:

In [ ]:
# Education
class Education(BaseModel):
    degree: str = Field(description="Degree or certification")
    institution: str = Field(description="University or school name")

# Experience
class Experience(BaseModel):
    job_title: str = Field(description="Job title")
    company: str = Field(description="Company name")

# Main Resume Model
class ResumeData(BaseModel):
    full_name: str = Field(description="Candidate full name")
    email: str = Field(description="Candidate email")
    phone: str = Field(description="Candidate phone number")
    address: str = Field(description="Candidate address")
    hobbies: List[str] = Field(description="List of hobbies")
    languages: List[str] = Field(description="Languages known")
    courses: List[str] = Field(description="Courses and certifications")
    education: List[Education] = Field(description="Education history")
    experience: List[Experience] = Field(description="Work experience")

* Extract and convert document using the Docling Vision Language Model (VLM) pipeline:

In [ ]:
source = "/content/resume.pdf"

converter = DocumentConverter(
    format_options={

        InputFormat.PDF: PdfFormatOption(
            pipeline_cls=VlmPipeline
        ),

        InputFormat.IMAGE: ImageFormatOption(
            pipeline_cls=VlmPipeline
        ),
    }
)

doc = converter.convert(source).document
markdown_output = doc.export_to_markdown()
print("MARKDOWN OUTPUT:\n", markdown_output)

Loading weights:   0%|          | 0/470 [00:00<?, ?it/s]

MARKDOWN OUTPUT:
 Logo

<!-- image -->

IT Consultant

## DETAILS

## ADDRESS

1515 Pacific Ave Los Angeles, CA 90291 United States

## PHONE

3868683442

## EMAIL

email@email.com

## PLACE OF BIRTH

San Antonio

## DRIVING LICENSE

Full

## LINKS

LinkedIn

Pinterest

Resume Templates

Build this template

## HOBBIES

Angling, Sailing, Fly Fishing

## LANGUAGES

English

French

## PROFILE

Personable IT Consultant with 5+ years of experience in a global technology firm. CompTIA A+ Certification. Scored 
the region leading QST rating based on internal reviews (97.86%), I am seeking to leverage solid technical skills 
and abilities to advance my career as the next IT consultant for Linsang Group.

## EMPLOYMENT HISTORY

## IT Consultant, Amazon

Jacksonville

Jan 2020-Jun 2021

Administer first-level MHE and PKMS support and under-provided SOPs to make appropriate corrections when necessary.

- · Researched and documented existing and new processes for IT Support Teams and interacted with business users 
and other IT groups to ascertain business requirements and design proposed system enhancements.
- · Communicated issues, resolutions, and the project status to IT management and user community and ensured the 
deadlines were met and quality was maximized.
- · Deployed, reset, configured, and replaced equipment as needed, such as CLI Terminals, Printers, Silex Printer 
boxes, CPUs and laptops.
- · Coached newly hired IT specialists on advanced technical procedures.

## IT Consultant, PWC

Pengcheng

Jan 2019-Dec 2021

Independent, a non-profit organization that provides a broad array of assessment, research, information, and 
program management solutions in the education and workforce development areas.

- · Identified software and hardware issues and listened to client concerns.
- · Encouraged timely and relevant upgrades for client products when necessary.
- · Devised a workable scheme to accomplish business objectives. Scheduled and allocated project activities, 
identified tools,
- · Provided risk management by monitoring project schedules. · Reported on a project's status regularly through 
emails and weekly
- meetings; formally tracked problems and issues to closure.

## EDUCATION

## Bachelor of Science in Information Systems Management, Miami University

Jan 2020-Jun 2021

- · Relevant Coursework: Network Security, IT Project Management, Business Administration, Strategy &amp; 
Operations, IT Innovation, Ethical Hacking, Database Management.

## COURSES

Microsoft Certified Solutions Expert, Microsoft. Online.

Jan 2020-Jun 2021

CCNA Routing and Switching, Cisco. Online.

Jan 2019-Aug 2019

## ACHIEVEMENTS

- · Identified a new parts-ordering solution which led to a reduced client wait time of 19% and an increase in 
client satisfaction by 41%
- · Assisted the IT director with administration applications, reducing the workload by 22%
- · Identified ticketing management solutions which led to a queue reduction of 21%
- · Assisted the IT manager as liaison to clients on software updates, reducing workload by over 52%

* Print The Structured Output:

In [ ]:
text = markdown_output

#  Remove image/logo placeholders
text = text.replace("LOGO", "")
text = text.replace("Logo", "")
text = text.replace("logo", "")
text = text.replace("<!-- image -->", "")

# Extract Full Name

lines = [line.strip() for line in text.splitlines() if line.strip()]
name_parts = []

for line in lines:

    if "consultant" in line.lower():
        break

    if line.isupper():
        name_parts.append(line)

    if len(name_parts) == 2:
        break

full_name = " ".join(name_parts)

# Extract Email

email_match = re.search(r"[\w\.-]+@[\w\.-]+", text)
email = email_match.group(0) if email_match else ""

# Extract Phone Number

phone_match = re.search(r"(\+?\d[\d\s\-]{7,})", text)
phone = phone_match.group(0).strip() if phone_match else ""

# Extract Address

address_match = re.search(r"\d{1,5}.*(?:Street|St|Road|Rd|Avenue|Ave|Boulevard|Blvd).*",text,re.IGNORECASE)
address = address_match.group(0) if address_match else ""

# Extract Hobbies

hobbies = []
hobbies_section = re.search(r"## HOBBIES(.*?)##", text, re.DOTALL | re.IGNORECASE)

if hobbies_section:
    hobby_lines = hobbies_section.group(1).split("\n")

    for hobby in hobby_lines:
        hobby = hobby.replace("-", "").strip()

        if hobby:
            parts = [h.strip() for h in hobby.split(",") if h.strip()]
            hobbies.extend(parts)

# Extract Languages

languages = []
languages_section = re.search(r"## LANGUAGES(.*?)##", text, re.DOTALL | re.IGNORECASE)

if languages_section:
    language_lines = languages_section.group(1).split("\n")

    for language in language_lines:
        language = language.replace("-", "").strip()

        if language:
            languages.append(language)

# Extract Courses

courses = []
course_section = re.search(r"# COURSES(.*?)(#|$)", text, re.DOTALL | re.IGNORECASE)

if course_section:
    for line in course_section.group(1).split("\n"):
        line = line.strip("-• ").strip()

        if not line:
            continue

        if re.match(r"^[A-Za-z]{3}\s?\d{4}", line):
            continue

        if re.match(r"^\w{3}\s?\d{4}\s?-\s?\w{3}\s?\d{4}$", line):
            continue

        courses.append(line)


# Extract Education

education = []
education_section = re.search(r"## EDUCATION(.*?)(## COURSES|$)", text, re.DOTALL | re.IGNORECASE)

if education_section:
    education_text = education_section.group(1)
    matches = re.findall(r"## (.*?), (.*?)\n", education_text)

    for degree, institution in matches:
        education.append(
            Education(
                degree=degree.strip(),
                institution=institution.strip()
            )
        )

# Extract Experience

experience = []
experience_section = re.search(r"## EMPLOYMENT HISTORY(.*?)(## EDUCATION|$)", text, re.DOTALL | re.IGNORECASE)

if experience_section:
    experience_text = experience_section.group(1)
    matches = re.findall(r"## (.*?), (.*?)\n", experience_text)

    for job_title, company in matches:
        experience.append(
            Experience(
                job_title=job_title.strip(),
                company=company.strip()
            )
        )


# Create Final Resume Object

resume_data = ResumeData(
    full_name=full_name,
    email=email,
    phone=phone,
    address=address,
    hobbies=hobbies,
    languages=languages,
    courses=courses,
    education=education,
    experience=experience,
)

# Print Structured JSON

print("\n========== STRUCTURED OUTPUT ==========\n")
print(resume_data.model_dump_json(indent=2))

========== STRUCTURED OUTPUT ==========

{
  "full_name": "",
  "email": "email@email.com",
  "phone": "3868683442",
  "address": "1515 Pacific Ave Los Angeles, CA 90291 United States",
  "hobbies": [
    "Angling",
    "Sailing",
    "Fly Fishing"
  ],
  "languages": [
    "English",
    "French"
  ],
  "courses": [
    "Microsoft Certified Solutions Expert, Microsoft. Online.",
    "CCNA Routing and Switching, Cisco. Online."
  ],
  "education": [
    {
      "degree": "Bachelor of Science in Information Systems Management",
      "institution": "Miami University"
    }
  ],
  "experience": [
    {
      "job_title": "IT Consultant",
      "company": "Amazon"
    },
    {
      "job_title": "IT Consultant",
      "company": "PWC"
    }
  ]
}

#### **form.docx:**

VLMs can only read documents as images, and DOCX is not an image format, we must convert it to PDF first.

* Pydantic Models:

In [ ]:
class DocumentSection(BaseModel):
    title: str = Field(description="Section title")
    content: List[str] = Field(description="Section content")

class DocumentData(BaseModel):
    title: str = Field(description="Document title")
    headings: List[str] = Field(description="Document headings")
    bullet_points: List[str] = Field(description="Bullet list items")
    numbered_points: List[str] = Field(description="Numbered list items")
    bold_texts: List[str] = Field(description="Bold text content")
    italic_texts: List[str] = Field(description="Italic text content")
    code_texts: List[str] = Field(description="Monospace/code text")
    blockquotes: List[str] = Field(description="Quoted text")
    sections: List[DocumentSection] = Field(description="Document sections")

* Extract and convert document using the Docling Vision Language Model (VLM) pipeline:


In [ ]:
source = "/content/sample document.pdf"

converter = DocumentConverter(
    format_options={

        InputFormat.PDF: PdfFormatOption(
            pipeline_cls=VlmPipeline
        ),

        InputFormat.IMAGE: ImageFormatOption(
            pipeline_cls=VlmPipeline
        ),
    }
)

doc = converter.convert(source).document
markdown_output = doc.export_to_markdown()
print("MARKDOWN OUTPUT:\n", markdown_output)

Loading weights:   0%|          | 0/470 [00:00<?, ?it/s]

MARKDOWN OUTPUT:
 ## Sample Document

This is a simple one-page document that demonstrates various text formatting options commonly found in DOCX files.

## Basic Text Formatting

Here is some text with bold formatting and italic formatting. You can also have bold and italic text combined.

## Lists

Here's a bulleted list:

- First item
- Second item
- Third item with some italic text

And a numbered list:

- First numbered item
- Second numbered item
- Third numbered item with bold text

## Different Sizes and Alignment

This paragraph demonstrates normal body text size and alignment. It contains multiple sentences to show how text 
flows in a typical document.

## Additional Formatting

You can also have:

monospace text for code or technical content

strikethrough text when needed

blockquotes for quoted content

* Print The Structured Output:

In [ ]:
text = markdown_output

# Clean placeholders
text = text.replace("<!-- image -->", "")
lines = [line.strip() for line in text.splitlines() if line.strip()]

# EXTRACT TITLE

title = lines[0] if lines else ""

# EXTRACT HEADINGS (##, ###)

headings = [line.replace("#", "").strip() for line in lines if line.startswith("#")]

# EXTRACT SECTIONS

sections = []
current_title = None
current_content = []

for line in lines:
    if line.startswith("## "):
        if current_title:
            sections.append(
                DocumentSection(
                    title=current_title,
                    content=current_content
                )
            )
        current_title = line.replace("##", "").strip()
        current_content = []
    else:
        if current_title:
            current_content.append(line)

if current_title:
    sections.append(
        DocumentSection(
            title=current_title,
            content=current_content
        )
    )

# BULLETED LISTS

bullet_points = []
for line in lines:
    if line.startswith("·") or line.startswith(". "):
        bullet_points.append(line.lstrip("·. ").strip())

# NUMBERED LISTS

numbered_points = []
for line in lines:
    if re.match(r"^\d+\.", line):
        numbered_points.append(line.split(".", 1)[1].strip())

# BOLD TEXT

bold_texts = re.findall(r"\*\*(.*?)\*\*", text)

# ITALIC TEXT

italic_texts = re.findall(r"\*(.*?)\*", text)

# CODE / MONOSPACE TEXT

code_texts = []
for line in lines:
    if "monospace" in line.lower():
        code_texts.append(line)

# BLOCKQUOTES

blockquotes = []
for line in lines:
    if "blockquote" in line.lower():
        blockquotes.append(line)

# CREATE FINAL SAMPLE DOCUMENT OBJECT

document_data = DocumentData(
    title=title,
    headings=headings,
    bullet_points=bullet_points,
    numbered_points=numbered_points,
    bold_texts=bold_texts,
    italic_texts=italic_texts,
    code_texts=code_texts,
    blockquotes=blockquotes,
    sections=sections,
)

print("\n========== STRUCTURED OUTPUT ==========\n")
print(document_data.model_dump_json(indent=2))

========== STRUCTURED OUTPUT ==========

{
  "title": "## Sample Document",
  "headings": [
    "Sample Document",
    "Basic Text Formatting",
    "Lists",
    "Different Sizes and Alignment",
    "Additional Formatting"
  ],
  "bullet_points": [],
  "numbered_points": [],
  "bold_texts": [],
  "italic_texts": [],
  "code_texts": [
    "monospace text for code or technical content"
  ],
  "blockquotes": [
    "blockquotes for quoted content"
  ],
  "sections": [
    {
      "title": "Sample Document",
      "content": [
        "This is a simple one-page document that demonstrates various text formatting options commonly found in 
DOCX files."
      ]
    },
    {
      "title": "Basic Text Formatting",
      "content": [
        "Here is some text with bold formatting and italic formatting. You can also have bold and italic text 
combined."
      ]
    },
    {
      "title": "Lists",
      "content": [
        "Here's a bulleted list:",
        "- First item",
        "- Second item",
        "- Third item with some italic text",
        "And a numbered list:",
        "- First numbered item",
        "- Second numbered item",
        "- Third numbered item with bold text"
      ]
    },
    {
      "title": "Different Sizes and Alignment",
      "content": [
        "This paragraph demonstrates normal body text size and alignment. It contains multiple sentences to show 
how text flows in a typical document."
      ]
    },
    {
      "title": "Additional Formatting",
      "content": [
        "You can also have:",
        "monospace text for code or technical content",
        "strikethrough text when needed",
        "blockquotes for quoted content"
      ]
    }
  ]
}